In [1]:
import pandas as pd
import numpy as np
import torch

from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm

d:\Guvi Projects\News Popularity Intelligence System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../outputs/clean_news.csv")

df.head()

,title,description,text,clean_text
0,Obama Lays Wreath at Arlington National Cemetery,Obama Lays Wreath at Arlington National Cemete...,Obama Lays Wreath at Arlington National Cemete...,obama lays wreath at arlington national cemete...
1,A Look at the Health of the Chinese Economy,"Tim Haywood, investment director business-unit...",A Look at the Health of the Chinese Economy Ti...,a look at the health of the chinese economy ti...
2,Nouriel Roubini: Global Economy Not Back to 2008,"Nouriel Roubini, NYU professor and chairman at...",Nouriel Roubini: Global Economy Not Back to 20...,nouriel roubini global economy not back to nou...
3,Finland GDP Expands In Q4,Finland's economy expanded marginally in the t...,Finland GDP Expands In Q4 Finland's economy ex...,finland gdp expands in q finlands economy expa...
4,"Tourism, govt spending buoys Thai economy in J...",Tourism and public spending continued to boost...,"Tourism, govt spending buoys Thai economy in J...",tourism govt spending buoys thai economy in ja...


In [3]:
df.shape

(216900, 4)

Load Tokenizer


In [4]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

Load Model

In [5]:
model = DistilBertModel.from_pretrained(
    "distilbert-base-uncased"
)

d:\Guvi Projects\News Popularity Intelligence System\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3087.57it/s]
DistilBertModel LOAD 

Test Tokenization

In [6]:
text = df["clean_text"][0]

tokens = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

tokens

{'input_ids': tensor([[  101,  8112, 19764, 29586,  2012, 13929,  2120,  4528,  8112, 19764,
         29586,  2012, 13929,  2120,  4528,  2343, 13857,  8112,  2038,  4201,
          1037, 29586,  2012,  1996,  8136,  1997,  1996,  4242,  2015,  2000,
          3932,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}

Pass Through Model

In [7]:
with torch.no_grad():

    outputs = model(**tokens)

outputs.last_hidden_state.shape

torch.Size([1, 32, 768])

Convert to Single Vector

In [8]:
embedding = outputs.last_hidden_state.mean(dim=1)

embedding.shape

torch.Size([1, 768])

Generate Embeddings for Dataset

⚠️ Use only 500 rows first (CPU safe)

In [9]:
embeddings = []

for text in tqdm(df["clean_text"][:500]):

    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**tokens)

    emb = outputs.last_hidden_state.mean(dim=1)

    emb = emb.squeeze().numpy()

    embeddings.append(emb)

100%|██████████| 500/500 [00:34<00:00, 14.69it/s]


Convert to Array

In [10]:
embeddings = np.array(embeddings)

embeddings.shape

(500, 768)

Save Embeddings

In [11]:
np.save("../outputs/embeddings.npy", embeddings)